<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Task 3</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Perform a forward and backward propagation in python using the inputs from Laboratory Task 2.
</p>

```python
x = np.array([1, 0, 1])
y = np.array([1])

# use relu as the activation function.

# learning rate
lr = 0.001
```
</div>

## Network Setup

Same architecture and weights as Laboratory Task 2: **3 inputs → 2 hidden units → 1 output unit**,
using ReLU, $f(Z) = \max(0, Z)$, as the activation function for every unit.

This follows the backpropagation formulas from the lecture notes, with the sigmoid derivative
$\sigma'(Z) = A(1-A)$ replaced by the **ReLU derivative**:

$$f'(Z) = \begin{cases} 1 & Z > 0 \\ 0 & Z \leq 0 \end{cases}$$

**Forward pass:**
$$Z = \sum_j (w_{i,j} \cdot x_i) + \theta_i \qquad A = f(Z)$$

**Backward pass (error signals):**
$$\delta_o = \hat{y} - y \qquad \delta_h = (\delta_o \, W_o^T) \cdot f'(Z_h)$$

**Gradients:**
$$\frac{\partial E}{\partial W_o} = A_h^T \, \delta_o \qquad \frac{\partial E}{\partial W_h} = X^T \, \delta_h$$

**Parameter update** (gradient descent, learning rate $\alpha$):
$$W = W - \alpha \frac{\partial E}{\partial W} \qquad \theta = \theta - \alpha \, \delta$$

In [1]:
import numpy as np

# --- Inputs (given) ---
x = np.array([1, 0, 1])
y = np.array([1])

# --- Learning rate (given) ---
lr = 0.001

# --- Weights & biases (from Laboratory Task 2) ---
# Hidden layer: rows = inputs x1,x2,x3 ; columns = hidden unit 1, hidden unit 2
W_h = np.array([
    [0.2, -0.3],   # w11, w12  (from x1)
    [0.4,  0.1],   # w13, w14  (from x2)
    [-0.5, 0.2]    # w15, w16  (from x3)
], dtype=float)

theta_h = np.array([-0.4, 0.2])   # theta1, theta2

# Output layer: hidden unit 1 -> output, hidden unit 2 -> output
W_o = np.array([-0.3, -0.2], dtype=float)
theta_o = 0.1                     # theta3

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

print("x =", x)
print("y =", y)
print("lr =", lr)

x = [1 0 1]
y = [1]
lr = 0.001


## 1. Forward Propagation

$$Z_h = x \cdot W_h + \theta_h \qquad A_h = f(Z_h)$$
$$Z_o = A_h \cdot W_o + \theta_o \qquad \hat{y} = f(Z_o)$$

In [2]:
# --- Hidden layer ---
Z_h = x @ W_h + theta_h
A_h = relu(Z_h)

# --- Output layer ---
Z_o = A_h @ W_o + theta_o
y_hat = relu(Z_o)

print("Z_h    =", Z_h)
print("A_h    =", A_h)
print("Z_o    =", Z_o)
print("y_hat  =", y_hat)

Z_h    = [-0.7  0.1]
A_h    = [0.  0.1]
Z_o    = 0.08
y_hat  = 0.08


## 2. Backward Propagation

**Output layer error signal:**
$$\delta_o = \hat{y} - y$$

**Hidden layer error signal**, propagated backward through the output weights and scaled by
the ReLU derivative at each hidden unit:
$$\delta_h = (\delta_o \, W_o^T) \cdot f'(Z_h)$$

In [3]:
# --- Output layer error signal ---
delta_o = y_hat - y

# --- Hidden layer error signal ---
delta_h = (delta_o * W_o) * relu_derivative(Z_h)

print("delta_o =", delta_o)
print("delta_h =", delta_h)

delta_o = [-0.92]
delta_h = [0.    0.184]


## 3. Gradients

$$\frac{\partial E}{\partial W_o} = A_h^T \, \delta_o \qquad \frac{\partial E}{\partial W_h} = X^T \, \delta_h$$

The bias gradients are simply the error signals themselves ($\delta_o$ for the output bias,
$\delta_h$ for the hidden biases).

In [4]:
# --- Gradient for output layer weights & bias ---
dW_o = A_h * delta_o          # A_h^T . delta_o  (outer product, single sample)
dtheta_o = delta_o

# --- Gradient for hidden layer weights & bias ---
dW_h = np.outer(x, delta_h)   # X^T . delta_h
dtheta_h = delta_h

print("dW_o     =", dW_o)
print("dtheta_o =", dtheta_o)
print("dW_h:\\n", dW_h)
print("dtheta_h =", dtheta_h)

dW_o     = [-0.    -0.092]
dtheta_o = [-0.92]
dW_h:\n [[0.    0.184]
 [0.    0.   ]
 [0.    0.184]]
dtheta_h = [0.    0.184]


## 4. Update Weights & Biases

$$W_o^{new} = W_o - \alpha \, \frac{\partial E}{\partial W_o} \qquad \theta_o^{new} = \theta_o - \alpha \, \delta_o$$
$$W_h^{new} = W_h - \alpha \, \frac{\partial E}{\partial W_h} \qquad \theta_h^{new} = \theta_h - \alpha \, \delta_h$$

In [5]:
W_o_new = W_o - lr * dW_o
theta_o_new = theta_o - lr * dtheta_o[0]

W_h_new = W_h - lr * dW_h
theta_h_new = theta_h - lr * dtheta_h

print("Updated W_o:", W_o_new)
print("Updated theta_o:", theta_o_new)
print("Updated W_h:\\n", W_h_new)
print("Updated theta_h:", theta_h_new)

Updated W_o: [-0.3      -0.199908]
Updated theta_o: 0.10092000000000001
Updated W_h:\n [[ 0.2      -0.300184]
 [ 0.4       0.1     ]
 [-0.5       0.199816]]
Updated theta_h: [-0.4       0.199816]


## 5. Verify: Forward Pass With Updated Weights

Running the forward pass again with the newly updated weights should give a prediction
slightly closer to the target $y = 1$.

In [6]:
Z_h_new = x @ W_h_new + theta_h_new
A_h_new = relu(Z_h_new)

Z_o_new = A_h_new @ W_o_new + theta_o_new
y_hat_new = relu(Z_o_new)

print("New y_hat      =", y_hat_new)
print("Previous y_hat =", y_hat)
print("Improved (closer to y=1)?", abs(y - y_hat_new)[0] < abs(y - y_hat)[0])

New y_hat      = 0.081039549216
Previous y_hat = 0.08
Improved (closer to y=1)? True


## Summary

| Quantity | Value |
|---|---|
| $\delta_o$ | -0.92 |
| $\delta_h$ | [0, 0.184] |
| $\partial E/\partial W_o$ | [0, -0.092] |
| $\partial E/\partial W_h$ | [[0, 0.184], [0, 0], [0, 0.184]] |
| $W_o$ (updated) | [-0.3, -0.199908] |
| $\theta_o$ (updated) | 0.10092 |
| $W_h$ (updated) | [[0.2, -0.300184], [0.4, 0.1], [-0.5, 0.199816]] |
| $\theta_h$ (updated) | [-0.4, 0.199816] |

Hidden unit 1's net input was negative ($Z_1 = -0.7$), so $f'(Z_1) = 0$ under ReLU — no error
signal reaches it, and its weights ($w_{11}, w_{13}, w_{15}$) and bias ($\theta_1$) are
**unchanged** after this update. Only the weights connected to hidden unit 2 (which was active)
and the output layer's weights/bias shift, each by a small amount since $\alpha = 0.001$.